# Exercise 05a — Tune and diagnose next-hour zone demand

Use the same Manhattan zone-hour data as Tutorial 05a, but investigate different modelling decisions. The data preparation is provided. Your work begins with validation-based tuning.

Complete Questions 1–3 without using test results. Freeze one setting per model, evaluate every frozen model once on test, then analyse where and when errors occur. Question 5 is the only extension: combine two already-tuned models without changing their internal parameters.

## Open this notebook

- **Colab.** Open the notebook, then run the setup cell below before any data-loading cell. It clones the public course repository and changes into `05 Transportation Applications I`, so relative data paths work.
- **On your own computer.** Open the notebook from its lesson folder so the data files sit beside it.


In [ ]:
from pathlib import Path
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    print("Running in Google Colab")
    REPO_URL = "https://github.com/hku-kejintao/AI-Code.git"
    LESSON_DIR = "05 Transportation Applications I"
    base = Path("/content")
    repo = base / "AI-Code"
    os.chdir(base)
    if not repo.exists():
        !git clone {REPO_URL}
    lesson_path = repo / LESSON_DIR
    if not lesson_path.exists():
        raise FileNotFoundError(f"Lesson folder not found: {lesson_path}")
    os.chdir(lesson_path)
else:
    print("Running on your own computer")

print("Current folder:", os.getcwd())


### Check the packages

The cell should print version numbers and the word `ready`. NumPy holds arrays, pandas holds tables, Matplotlib draws figures, and scikit-learn provides preprocessing, models, and metrics.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
print("ready")


### Install, only if the check failed

Skip this cell when the check printed `ready`. After installation finishes, restart the kernel and run the cells above again.


In [ ]:
%pip install numpy pandas matplotlib scikit-learn


## Provided data preparation

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})

required = ["nyc_zone_hour.csv.gz", "nyc_zone_features.csv", "nyc_taxi_zones.geojson"]
assert all(Path(name).exists() for name in required), "Run the notebook from the Module 05 folder."

flow = pd.read_csv("nyc_zone_hour.csv.gz", parse_dates=["time"])
zones = pd.read_csv("nyc_zone_features.csv")
geo = json.load(open("nyc_taxi_zones.geojson"))

def polygons(feature):
    geometry = feature["geometry"]
    return geometry["coordinates"] if geometry["type"] == "MultiPolygon" else [geometry["coordinates"]]

def zone_map(values, title, cmap="magma", ax=None):
    values = pd.Series(values)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))
    norm = plt.Normalize(values.min(), values.max() if values.max() > values.min() else values.min() + 1)
    for feature in geo["features"]:
        zone_id = int(feature["properties"]["LocationID"])
        value = values.get(zone_id, np.nan)
        colour = "lightgrey" if pd.isna(value) else plt.colormaps[cmap](norm(value))
        for polygon in polygons(feature):
            ax.fill(*np.asarray(polygon[0]).T, facecolor=colour, edgecolor="white", linewidth=.45)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=.7)
    return ax

d = flow.sort_values(["zone_id", "time"]).copy()
for lag in [1, 2, 3, 24, 168]:
    d[f"lag_{lag}"] = d.groupby("zone_id")["pickups"].shift(lag)
d["hour_sin"] = np.sin(2 * np.pi * d.time.dt.hour / 24)
d["hour_cos"] = np.cos(2 * np.pi * d.time.dt.hour / 24)
d["dow_sin"] = np.sin(2 * np.pi * d.time.dt.dayofweek / 7)
d["dow_cos"] = np.cos(2 * np.pi * d.time.dt.dayofweek / 7)
d["weekend"] = (d.time.dt.dayofweek >= 5).astype(int)
d = d.merge(
    zones[["LocationID", "population_density", "employment_density", "poi_total"]],
    left_on="zone_id",
    right_on="LocationID",
).dropna().reset_index(drop=True)

features = [
    "lag_1", "lag_2", "lag_3", "lag_24", "lag_168",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "weekend",
    "population_density", "employment_density", "poi_total",
]
train = d.time < pd.Timestamp("2024-02-10")
valid = d.time.between("2024-02-10", "2024-02-20", inclusive="left")
test = d.time >= pd.Timestamp("2024-02-20")
y = d["pickups"].to_numpy()

X_train = d.loc[train, features].to_numpy()
y_train = y[train]
X_valid = d.loc[valid, features].to_numpy()
y_valid = y[valid]
X_test = d.loc[test, features].to_numpy()
y_test = y[test]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)
print("Each row describes one named taxi zone at one target hour.")

def scores(name, observed, predicted):
    return {
        "model": name,
        "MAE": mean_absolute_error(observed, predicted),
        "RMSE": mean_squared_error(observed, predicted) ** .5,
        "R2": r2_score(observed, predicted),
    }

## Question 1 — Tune Ridge regularisation

Fit Ridge models with `alpha` in `[0.01, 0.1, 1, 10, 100]`. Standardise the input features inside a pipeline, calculate validation MAE, plot MAE against `alpha` on a log scale, and freeze the best `alpha`. Explain why the test period must remain untouched.

In [ ]:
# TODO: fit the five Ridge candidates on X_train and y_train.
# TODO: calculate validation MAE, plot the sensitivity curve, and store best_ridge_alpha.

## Question 2 — Tune forest depth and leaf size

For a random forest, compare `max_depth` in `[10, None]` and `min_samples_leaf` in `[1, 3, 8]`. Keep all other settings fixed. Use validation MAE to freeze one pair of settings, then describe the bias–variance pattern visible in the table.

In [ ]:
# TODO: evaluate the 2 × 3 forest grid on validation data.
# Store the winning pair as best_forest_key = (max_depth, min_samples_leaf).

## Question 3 — Freeze four model families and compare once on test

Tune ANN width in `[16, 32, 64]` using validation MAE. Then refit Linear Regression, the selected Ridge, the selected forest, and the selected ANN on train + validation. Evaluate all four models on test once using MAE, RMSE, and $R^2$. Plot the final comparison.

In [ ]:
# TODO: tune ANN width on validation data.
# TODO: refit Linear, selected Ridge, selected forest, and selected ANN on train + validation.
# TODO: evaluate every frozen model once on X_test and y_test.

## Question 4 — Locate difficult places and times

For the test-best frozen model, draw (a) a Manhattan map of zone-level MAE and (b) MAE by hour of day. Also plot observed and predicted demand for the busiest test-period zone. Identify at least one spatial and one temporal error pattern, then suggest one additional feature that could address each pattern.

In [ ]:
# TODO: aggregate absolute test error by zone and by hour.
# TODO: call zone_map, draw the hourly curve, and plot the busiest named zone.
# Write a short diagnosis below the figures.

## Question 5 — Extension: validation-selected model blending

Blend the selected forest and ANN predictions as

$$\hat y=w\hat y_{RF}+(1-w)\hat y_{ANN}.$$

Choose $w$ from `[0, 0.25, 0.5, 0.75, 1]` using validation MAE only. Freeze the weight, combine the already-fitted test predictions, and add the blend to the final test table. Explain why this is a new model-selection decision even though neither base model is retrained.

In [ ]:
# TODO: choose the forest weight using validation predictions only.
# TODO: combine the frozen test predictions once and append the blend to the final table.